# **Youtube Videos Transcription with OpenAI's Whisper**

[![blog post shield](https://img.shields.io/static/v1?label=&message=Blog%20post&color=blue&style=for-the-badge&logo=openai&link=https://openai.com/blog/whisper)](https://openai.com/blog/whisper)
[![notebook shield](https://img.shields.io/static/v1?label=&message=Notebook&color=blue&style=for-the-badge&logo=googlecolab&link=https://colab.research.google.com/github/ArthurFDLR/whisper-youtube/blob/main/whisper_youtube.ipynb)](https://colab.research.google.com/github/ArthurFDLR/whisper-youtube/blob/main/whisper_youtube.ipynb)
[![repository shield](https://img.shields.io/static/v1?label=&message=Repository&color=blue&style=for-the-badge&logo=github&link=https://github.com/openai/whisper)](https://github.com/openai/whisper)
[![paper shield](https://img.shields.io/static/v1?label=&message=Paper&color=blue&style=for-the-badge&link=https://cdn.openai.com/papers/whisper.pdf)](https://cdn.openai.com/papers/whisper.pdf)
[![model card shield](https://img.shields.io/static/v1?label=&message=Model%20card&color=blue&style=for-the-badge&link=https://github.com/openai/whisper/blob/main/model-card.md)](https://github.com/openai/whisper/blob/main/model-card.md)

Whisper is a general-purpose speech recognition model. It is trained on a large dataset of diverse audio and is also a multi-task model that can perform multilingual speech recognition as well as speech translation and language identification.

This Notebook will guide you through the transcription of a Youtube video using Whisper. You'll be able to explore most inference parameters or use the Notebook as-is to store the transcript and video audio in your Google Drive.

In [ ]:
#@markdown # **Check GPU type** 🕵️

#@markdown The type of GPU you get assigned in your Colab session defined the speed at which the video will be transcribed.
#@markdown The higher the number of floating point operations per second (FLOPS), the faster the transcription.
#@markdown But even the least powerful GPU available in Colab is able to run any Whisper model.
#@markdown Make sure you've selected `GPU` as hardware accelerator for the Notebook (Runtime &rarr; Change runtime type &rarr; Hardware accelerator).

#@markdown |  GPU   |  GPU RAM   | FP32 teraFLOPS |     Availability   |
#@markdown |:------:|:----------:|:--------------:|:------------------:|
#@markdown |  T4    |    16 GB   |       8.1      |         Free       |
#@markdown | P100   |    16 GB   |      10.6      |      Colab Pro     |
#@markdown | V100   |    16 GB   |      15.7      |  Colab Pro (Rare)  |

#@markdown ---
#@markdown **Factory reset your Notebook's runtime if you want to get assigned a new GPU.**

!nvidia-smi -L

!nvidia-smi

In [ ]:
#@markdown # **Install libraries** 🏗️
#@markdown This cell will take a little while to download several libraries, including Whisper.

#@markdown ---

! pip install git+https://github.com/openai/whisper.git
! pip install -U "yt-dlp[default]"
! test -x /root/.deno/bin/deno || curl -fsSL https://deno.land/install.sh | sh

import os
os.environ["PATH"] = "/root/.deno/bin:" + os.environ.get("PATH", "")

import sys
import warnings
import whisper
from pathlib import Path
import yt_dlp
import subprocess
import torch
import shutil
import numpy as np
from IPython.display import display, Markdown, YouTubeVideo

device = torch.device('cuda:0')
print('Using device:', device, file=sys.stderr)

In [ ]:
#@markdown # **Optional:** Save data in Google Drive 💾
#@markdown Enter a Google Drive path and run this cell if you want to store the results inside Google Drive.

# Uncomment to copy generated images to drive, faster than downloading directly from colab in my experience.
from google.colab import drive
drive_mount_path = Path("/") / "content" / "drive"
drive.mount(str(drive_mount_path))
drive_mount_path /= "My Drive"
#@markdown ---
drive_path = "Colab Notebooks/Whisper Youtube" #@param {type:"string"}
#@markdown ---
#@markdown **Run this cell again if you change your Google Drive path.**

drive_whisper_path = drive_mount_path / Path(drive_path.lstrip("/"))
drive_whisper_path.mkdir(parents=True, exist_ok=True)

In [ ]:
#@markdown # **Model selection** 🧠

#@markdown As of the first public release, there are 4 pre-trained options to play with:

#@markdown |  Size  | Parameters | English-only model | Multilingual model | Required VRAM | Relative speed |
#@markdown |:------:|:----------:|:------------------:|:------------------:|:-------------:|:--------------:|
#@markdown |  tiny  |    39 M    |     `tiny.en`      |       `tiny`       |     ~1 GB     |      ~32x      |
#@markdown |  base  |    74 M    |     `base.en`      |       `base`       |     ~1 GB     |      ~16x      |
#@markdown | small  |   244 M    |     `small.en`     |      `small`       |     ~2 GB     |      ~6x       |
#@markdown | medium |   769 M    |    `medium.en`     |      `medium`      |     ~5 GB     |      ~2x       |
#@markdown | large  |   1550 M   |        N/A         |      `large`       |    ~10 GB     |       1x       |

#@markdown ---
Model = 'medium' #@param ['tiny.en', 'tiny', 'base.en', 'base', 'small.en', 'small', 'medium.en', 'medium', 'large']
#@markdown ---
#@markdown **Run this cell again if you change the model.**

whisper_model = whisper.load_model(Model)

if Model in whisper.available_models():
    display(Markdown(
        f"**{Model} model is selected.**"
    ))
else:
    display(Markdown(
        f"**{Model} model is no longer available.**<br /> Please select one of the following:<br /> - {'<br /> - '.join(whisper.available_models())}"
    ))

In [ ]:
#@markdown # **Video selection** 📺

#@markdown Enter one or more Youtube URLs, a Google Drive path, or a shared Google Drive file link, and run the cell.

Type = "Youtube video or playlist" #@param ['Youtube video or playlist', 'Google Drive']
#@markdown ---
#@markdown #### **Youtube video or playlist**
URL = "https://youtu.be/L_Guz73e6fw" #@param {type:"string"}
#@markdown Paste multiple URLs separated by commas or line breaks. Playlist URLs are also supported.
youtube_cookies_file = "" #@param {type:"string"}
#@markdown Optional: Drive/local path to a Netscape `cookies.txt` file. If blank or missing, Colab will ask you to upload it only if YouTube requires sign-in.
# store_audio = True #@param {type:"boolean"}
#@markdown ---
#@markdown #### **Google Drive video, audio (mp4, wav), folder, or shared file link**
video_path = "Colab Notebooks/transcription/my_video.mp4" #@param {type:"string"}
#@markdown ---
#@markdown **Run this cell again if you change the video queue.**

import re
import unicodedata

video_path_local_list = []


def is_google_drive_url(value):
    return isinstance(value, str) and "drive.google.com" in value


def extract_google_drive_file_id(url):
    patterns = [r"/file/d/([^/]+)", r"[?&]id=([^&]+)", r"/d/([^/]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract Google Drive file id from: {url}")


def filename_slug(value, fallback="transcription"):
    value = unicodedata.normalize("NFKD", str(value or fallback)).encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^A-Za-z0-9]+", "-", value).strip("-").lower()
    return value or fallback


def unique_path(path):
    path = Path(path)
    if not path.exists():
        return path

    for index in range(2, 1000):
        candidate = path.with_name(f"{path.stem}-{index}{path.suffix}")
        if not candidate.exists():
            return candidate

    raise FileExistsError(f"Could not create a unique file name for {path}")


def split_video_queue(value):
    if isinstance(value, (list, tuple)):
        items = value
    else:
        items = re.split(r"[\n,]+", str(value or ""))
    return [item.strip() for item in items if item and item.strip()]


def ensure_drive_mount():
    global drive_mount_path, drive_whisper_path

    if "drive_mount_path" in globals() and drive_mount_path.exists():
        return drive_mount_path

    from google.colab import drive
    drive_root = Path("/") / "content" / "drive"
    drive.mount(str(drive_root))
    drive_mount_path = drive_root / "My Drive"

    if "drive_path" in globals():
        drive_whisper_path = drive_mount_path / Path(drive_path.lstrip("/"))
        drive_whisper_path.mkdir(parents=True, exist_ok=True)

    return drive_mount_path


def resolve_colab_or_drive_path(value, mount_drive=False):
    raw_value = str(value or "").strip()
    path = Path(raw_value).expanduser()
    if path.exists():
        return path.resolve()

    content_candidate = Path("/content") / Path(raw_value.lstrip("/"))
    if content_candidate.exists():
        return content_candidate

    if mount_drive or "drive_mount_path" in globals():
        drive_root = ensure_drive_mount()
        relative_value = raw_value.removeprefix("My Drive/").removeprefix("MyDrive/")
        drive_candidate = drive_root / Path(relative_value.lstrip("/"))
        if drive_candidate.exists():
            return drive_candidate

    return path


def find_cookie_file_by_name(file_name):
    if not file_name:
        return None

    search_roots = [Path("/content")]
    if "drive_whisper_path" in globals():
        search_roots.append(drive_whisper_path)
    if "drive_mount_path" in globals():
        search_roots.append(drive_mount_path)

    for root in search_roots:
        if not root.exists():
            continue
        direct_candidate = root / file_name
        if direct_candidate.is_file():
            return direct_candidate
        for candidate in root.rglob(file_name):
            if candidate.is_file():
                return candidate

    return None


def upload_cookie_file():
    from google.colab import files

    display(Markdown("**Upload your YouTube cookies.txt file to continue.**"))
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No cookies file was uploaded.")

    uploaded_name = next(iter(uploaded.keys()))
    uploaded_path = Path("/content") / uploaded_name
    if not uploaded_path.is_file():
        raise FileNotFoundError(f"Uploaded cookies file not found: {uploaded_path}")

    return uploaded_path


def register_local_media(source_path, output_stem=None):
    source_path = Path(source_path)
    target_stem = filename_slug(output_stem or source_path.stem)
    target_path = unique_path(Path(".").resolve() / f"{target_stem}{source_path.suffix.lower()}")

    if source_path.resolve() != target_path.resolve():
        shutil.copy(source_path, target_path)

    video_path_local_list.append(target_path)
    display(Markdown(f"**{source_path} selected as `{target_path.name}`.**"))
    return target_path


def download_shared_drive_file(url):
    import signal
    from google.colab import auth
    from google.auth import default
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    def timeout_handler(_signum, _frame):
        raise TimeoutError(
            "Google Drive authorization did not complete. If no popup appears, "
            "add a shortcut/copy of the file to My Drive and use its Drive path instead."
        )

    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(60)
    try:
        auth.authenticate_user()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    creds, _ = default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    drive_service = build("drive", "v3", credentials=creds)
    file_id = extract_google_drive_file_id(url)
    metadata = drive_service.files().get(fileId=file_id, fields="id,name,mimeType,size").execute()

    suffix = Path(metadata["name"]).suffix or ".mp4"
    video_path_local = unique_path(Path(".").resolve() / f"{filename_slug(Path(metadata['name']).stem)}{suffix.lower()}")
    display(Markdown(f"**Downloading shared Drive file {metadata['name']} for transcription.**"))

    request = drive_service.files().get_media(fileId=file_id)
    with video_path_local.open("wb") as file_handle:
        downloader = MediaIoBaseDownload(file_handle, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Google Drive download: {int(status.progress() * 100)}%")

    video_path_local_list.append(video_path_local)
    return video_path_local


def youtube_cookiefile_path(value, allow_upload=False):
    raw_value = str(value or "").strip()

    if raw_value:
        path = resolve_colab_or_drive_path(raw_value, mount_drive=True)
        if path.is_file():
            return str(path)

        found_path = find_cookie_file_by_name(Path(raw_value).name)
        if found_path:
            display(Markdown(f"**Using cookies file: `{found_path}`**"))
            return str(found_path)

        display(Markdown(f"**Cookies file not found at `{raw_value}`.**"))

    if allow_upload:
        return str(upload_cookie_file())

    return None


def youtube_needs_cookies(error):
    message = str(error).lower()
    return any(text in message for text in ("sign in", "not a bot", "cookies", "confirm you"))


def youtube_video_unavailable(error):
    message = str(error).lower()
    return any(text in message for text in ("not available", "unavailable", "private video", "video unavailable"))


def collect_youtube_downloads(info):
    if not info:
        return []

    if "entries" in info:
        downloads = []
        for entry in info.get("entries") or []:
            downloads.extend(collect_youtube_downloads(entry))
        return downloads

    source_path = Path(f"{info['id']}.wav").resolve()
    target_path = unique_path(source_path.with_name(f"{filename_slug(info.get('title'), info['id'])}.wav"))
    if source_path.exists() and source_path != target_path:
        source_path.rename(target_path)
    elif not target_path.exists():
        target_path = source_path

    return [target_path]


if Type == "Youtube video or playlist":
    youtube_urls = split_video_queue(URL)
    if not youtube_urls:
        raise ValueError("Please provide at least one YouTube URL.")

    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': '%(id)s.%(ext)s',
        'noplaylist': False,
        'remote_components': ['ejs:github'],
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
        }],
    }

    cookiefile = youtube_cookiefile_path(youtube_cookies_file)
    if cookiefile:
        ydl_opts['cookiefile'] = cookiefile

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for youtube_url in youtube_urls:
            try:
                video_info = ydl.extract_info(youtube_url, download=True)
            except yt_dlp.utils.DownloadError as error:
                if youtube_needs_cookies(error) and not ydl_opts.get('cookiefile'):
                    cookiefile = youtube_cookiefile_path(youtube_cookies_file, allow_upload=True)
                    ydl_opts['cookiefile'] = cookiefile
                    display(Markdown("**Retrying YouTube download with uploaded cookies.**"))
                    with yt_dlp.YoutubeDL(ydl_opts) as authenticated_ydl:
                        video_info = authenticated_ydl.extract_info(youtube_url, download=True)
                elif youtube_video_unavailable(error):
                    display(Markdown(f"**Skipping unavailable YouTube video:** `{youtube_url}`"))
                    continue
                else:
                    raise

            video_path_local_list.extend(collect_youtube_downloads(video_info))

elif Type == "Google Drive":
    if is_google_drive_url(video_path):
        download_shared_drive_file(video_path)
    else:
        video_path = ensure_drive_mount() / Path(video_path.lstrip("/"))
        if video_path.is_dir():
            for video_path_drive in video_path.glob("**/*"):
                if video_path_drive.is_file():
                    register_local_media(video_path_drive)
                elif video_path_drive.is_dir():
                    display(Markdown(f"**Subfolders not supported.**"))
                else:
                    display(Markdown(f"**{str(video_path_drive)} does not exist, skipping.**"))
        elif video_path.is_file():
            register_local_media(video_path)
        else:
            display(Markdown(f"**{str(video_path)} does not exist.**"))

else:
    raise(TypeError("Please select supported input type."))

converted_video_path_local_list = []
for video_path_local in video_path_local_list:
    if video_path_local.suffix.lower() == ".mp4":
        wav_path_local = video_path_local.with_suffix(".wav")
        result = subprocess.run([
            "ffmpeg", "-y", "-i", str(video_path_local), "-vn", "-acodec", "pcm_s16le",
            "-ar", "16000", "-ac", "1", str(wav_path_local)
        ], check=True)
        converted_video_path_local_list.append(wav_path_local)
    else:
        converted_video_path_local_list.append(video_path_local)

video_path_local_list = converted_video_path_local_list

if video_path_local_list:
    display(Markdown("**Transcription queue:**"))
    for index, video_path_local in enumerate(video_path_local_list, start=1):
        display(Markdown(f"{index}. `{video_path_local.name}`"))

In [ ]:
#@markdown # **Run the model** 🚀

#@markdown Run this cell to execute the transcription of the video. This can take a while and very based on the length of the video and the number of parameters of the model selected above.

#@markdown ## **Parameters** ⚙️

#@markdown ### **Behavior control**
#@markdown ---
language = "English" #@param ['Auto detection', 'Afrikaans', 'Albanian', 'Amharic', 'Arabic', 'Armenian', 'Assamese', 'Azerbaijani', 'Bashkir', 'Basque', 'Belarusian', 'Bengali', 'Bosnian', 'Breton', 'Bulgarian', 'Burmese', 'Castilian', 'Catalan', 'Chinese', 'Croatian', 'Czech', 'Danish', 'Dutch', 'English', 'Estonian', 'Faroese', 'Finnish', 'Flemish', 'French', 'Galician', 'Georgian', 'German', 'Greek', 'Gujarati', 'Haitian', 'Haitian Creole', 'Hausa', 'Hawaiian', 'Hebrew', 'Hindi', 'Hungarian', 'Icelandic', 'Indonesian', 'Italian', 'Japanese', 'Javanese', 'Kannada', 'Kazakh', 'Khmer', 'Korean', 'Lao', 'Latin', 'Latvian', 'Letzeburgesch', 'Lingala', 'Lithuanian', 'Luxembourgish', 'Macedonian', 'Malagasy', 'Malay', 'Malayalam', 'Maltese', 'Maori', 'Marathi', 'Moldavian', 'Moldovan', 'Mongolian', 'Myanmar', 'Nepali', 'Norwegian', 'Nynorsk', 'Occitan', 'Panjabi', 'Pashto', 'Persian', 'Polish', 'Portuguese', 'Punjabi', 'Pushto', 'Romanian', 'Russian', 'Sanskrit', 'Serbian', 'Shona', 'Sindhi', 'Sinhala', 'Sinhalese', 'Slovak', 'Slovenian', 'Somali', 'Spanish', 'Sundanese', 'Swahili', 'Swedish', 'Tagalog', 'Tajik', 'Tamil', 'Tatar', 'Telugu', 'Thai', 'Tibetan', 'Turkish', 'Turkmen', 'Ukrainian', 'Urdu', 'Uzbek', 'Valencian', 'Vietnamese', 'Welsh', 'Yiddish', 'Yoruba']
#@markdown > Language spoken in the audio, use `Auto detection` to let Whisper detect the language.
#@markdown ---
verbose = 'Live transcription' #@param ['Live transcription', 'Progress bar', 'None']
#@markdown > Whether to print out the progress and debug messages.
#@markdown ---
output_format = 'all' #@param ['txt', 'vtt', 'srt', 'tsv', 'json', 'all']
#@markdown > Type of file to generate to record the transcription.
#@markdown ---
task = 'transcribe' #@param ['transcribe', 'translate']
#@markdown > Whether to perform X->X speech recognition (`transcribe`) or X->English translation (`translate`).
#@markdown ---

#@markdown <br/>

#@markdown ### **Optional: Fine tunning**
#@markdown ---
temperature = 0.15 #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown > Temperature to use for sampling.
#@markdown ---
temperature_increment_on_fallback = 0.2 #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown > Temperature to increase when falling back when the decoding fails to meet either of the thresholds below.
#@markdown ---
best_of = 5 #@param {type:"integer"}
#@markdown > Number of candidates when sampling with non-zero temperature.
#@markdown ---
beam_size = 8 #@param {type:"integer"}
#@markdown > Number of beams in beam search, only applicable when temperature is zero.
#@markdown ---
patience = 1.0 #@param {type:"number"}
#@markdown > Optional patience value to use in beam decoding, as in [*Beam Decoding with Controlled Patience*](https://arxiv.org/abs/2204.05424), the default (1.0) is equivalent to conventional beam search.
#@markdown ---
length_penalty = -0.05 #@param {type:"slider", min:-0.05, max:1, step:0.05}
#@markdown > Optional token length penalty coefficient (alpha) as in [*Google's Neural Machine Translation System*](https://arxiv.org/abs/1609.08144), set to negative value to uses simple length normalization.
#@markdown ---
suppress_tokens = "-1" #@param {type:"string"}
#@markdown > Comma-separated list of token ids to suppress during sampling; '-1' will suppress most special characters except common punctuations.
#@markdown ---
initial_prompt = "" #@param {type:"string"}
#@markdown > Optional text to provide as a prompt for the first window.
#@markdown ---
condition_on_previous_text = True #@param {type:"boolean"}
#@markdown > if True, provide the previous output of the model as a prompt for the next window; disabling may make the text inconsistent across windows, but the model becomes less prone to getting stuck in a failure loop.
#@markdown ---
fp16 = True #@param {type:"boolean"}
#@markdown > whether to perform inference in fp16.
#@markdown ---
compression_ratio_threshold = 2.4 #@param {type:"number"}
#@markdown > If the gzip compression ratio is higher than this value, treat the decoding as failed.
#@markdown ---
logprob_threshold = -1.0 #@param {type:"number"}
#@markdown > If the average log probability is lower than this value, treat the decoding as failed.
#@markdown ---
no_speech_threshold = 0.6 #@param {type:"slider", min:-0.0, max:1, step:0.05}
#@markdown > If the probability of the <|nospeech|> token is higher than this value AND the decoding has failed due to `logprob_threshold`, consider the segment as silence.
#@markdown ---

verbose_lut = {
    'Live transcription': True,
    'Progress bar': False,
    'None': None
}

args = dict(
    language = (None if language == "Auto detection" else language),
    verbose = verbose_lut[verbose],
    task = task,
    temperature = temperature,
    temperature_increment_on_fallback = temperature_increment_on_fallback,
    best_of = best_of,
    beam_size = beam_size,
    patience=patience,
    length_penalty=(length_penalty if length_penalty>=0.0 else None),
    suppress_tokens=suppress_tokens,
    initial_prompt=(None if not initial_prompt else initial_prompt),
    condition_on_previous_text=condition_on_previous_text,
    fp16=fp16,
    compression_ratio_threshold=compression_ratio_threshold,
    logprob_threshold=logprob_threshold,
    no_speech_threshold=no_speech_threshold
)

temperature = args.pop("temperature")
temperature_increment_on_fallback = args.pop("temperature_increment_on_fallback")
if temperature_increment_on_fallback is not None:
    temperature = tuple(np.arange(temperature, 1.0 + 1e-6, temperature_increment_on_fallback))
else:
    temperature = [temperature]

if Model.endswith(".en") and args["language"] not in {"en", "English"}:
    warnings.warn(f"{Model} is an English-only model but receipted '{args['language']}'; using English instead.")
    args["language"] = "en"

for video_path_local in video_path_local_list:
    display(Markdown(f"### {video_path_local}"))

    video_transcription = whisper.transcribe(
        whisper_model,
        str(video_path_local),
        temperature=temperature,
        **args,
    )

    # Save output
    whisper.utils.get_writer(
        output_format=output_format,
        output_dir=video_path_local.parent
    )(
        video_transcription,
        str(video_path_local.stem),
        options=dict(
            highlight_words=False,
            max_line_count=None,
            max_line_width=None,
        )
    )

    def exportTranscriptFile(ext: str):
        local_path = video_path_local.parent / video_path_local.with_suffix(ext).name
        export_path = drive_whisper_path / video_path_local.with_suffix(ext).name
        shutil.copy(
            local_path,
            export_path
        )
        display(Markdown(f"**Transcript file created: {export_path}**"))

    if output_format=="all":
        for ext in ('.txt', '.vtt', '.srt', '.tsv', '.json'):
            exportTranscriptFile(ext)
    else:
        exportTranscriptFile("." + output_format)
